### Find Failed Drives

Loop through each of the daily CSV files and filter for rows where `failure == 1` and combine those rows into one DataFrame.

This gives the known failure dates and serial numbers that will be used to create the `failure_within_30_days` target.

In [ ]:
import pandas as pd
from pathlib import Path

file_paths = Path("../data/raw").glob("*.csv")
result = []

for file_path in file_paths:
  current_df = pd.read_csv(file_path)
  current_df = current_df[current_df['failure'] == 1]
  result.append(current_df)

failed_df = pd.concat(result, ignore_index=True)
print(failed_df)


            date serial_number                 model  capacity_bytes  failure  \
0     2026-01-01      ZHZ3X3YY         ST12000NM0008  12000138625024        1   
1     2026-01-02      2BJBEESN   WDC WUH721816ALE6L4  16000900661248        1   
2     2026-01-02  40P0A0F1F97G   TOSHIBA MG07ACA14TA  14000519643136        1   
3     2026-01-02      8CGKHZ5G  HGST HUH721212ALE600  12000138625024        1   
4     2026-01-03      1PG8S6AV   WDC WUH722222ALE6L4  22000969973760        1   
...          ...           ...                   ...             ...      ...   
1025  2026-03-31      ZA181A8X          ST8000NM0055   8001563222016        1   
1026  2026-03-31      ZHZ50R9S         ST12000NM0008  12000138625024        1   
1027  2026-03-31      ZL22YS77         ST16000NM001G  16000900661248        1   
1028  2026-03-31      ZL23X50P         ST16000NM001G  16000900661248        1   
1029  2026-03-31      ZL2CV1VC         ST14000NM001G  14000519643136        1   

     datacenter  cluster_id

In [9]:
failure_dates = failed_df[['date', 'serial_number']]

print(failure_dates)

            date serial_number
0     2026-01-01      ZHZ3X3YY
1     2026-01-02      2BJBEESN
2     2026-01-02  40P0A0F1F97G
3     2026-01-02      8CGKHZ5G
4     2026-01-03      1PG8S6AV
...          ...           ...
1025  2026-03-31      ZA181A8X
1026  2026-03-31      ZHZ50R9S
1027  2026-03-31      ZL22YS77
1028  2026-03-31      ZL23X50P
1029  2026-03-31      ZL2CV1VC

[1030 rows x 2 columns]


In [16]:
failure_dates_copy = failure_dates.copy()

unique_serial = failed_df['serial_number'].unique()
failure_dates_copy['date'] = failure_dates_copy['date'].astype('datetime64[ns]')

print(len(unique_serial))
print(failure_dates_copy['date'].dtype)

1030
datetime64[ns]


### Collect Failed Drive History

Use the serial numbers from `failure_dates` to find all the previous observations for drives that eventually failed.

This creates a history of each failed drive's S.M.A.R.T. telemetry leading up to their failure. These observations will be compared with the drive's known failure date later on to calculate how many days remained until failure.

In [23]:
file_paths = Path("../data/raw").glob("*.csv")
result = []

for file_path in file_paths: 
  current_df = pd.read_csv(file_path)
  current_df = current_df[current_df['serial_number'].isin(failure_dates['serial_number'])]
  result.append(current_df)

failed_history = pd.concat(result, ignore_index=True)
print(failed_history)

             date serial_number                 model  capacity_bytes  \
0      2026-01-01  1080A183F9RG  TOSHIBA MG07ACA14TEY  14000519643136   
1      2026-01-01  10B0A0VUF97G   TOSHIBA MG07ACA14TA  14000519643136   
2      2026-01-01  10F0A00EF9RG  TOSHIBA MG07ACA14TEY  14000519643136   
3      2026-01-01  10G0A004F97G   TOSHIBA MG07ACA14TA  14000519643136   
4      2026-01-01  10J0A0D9F97G   TOSHIBA MG07ACA14TA  14000519643136   
...           ...           ...                   ...             ...   
43148  2026-03-31      ZA181A8X          ST8000NM0055   8001563222016   
43149  2026-03-31      ZHZ50R9S         ST12000NM0008  12000138625024   
43150  2026-03-31      ZL22YS77         ST16000NM001G  16000900661248   
43151  2026-03-31      ZL23X50P         ST16000NM001G  16000900661248   
43152  2026-03-31      ZL2CV1VC         ST14000NM001G  14000519643136   

       failure datacenter  cluster_id  vault_id  pod_id  pod_slot_num  ...  \
0            0       sac0           0      11